# Exploring the GIS test data (permits, service requests, road segments)

There is no data dictionary, so before setting up the ArcGIS Pro project I checked the three layers for the decisions the geoprocessing depends on:

1. **Load:** what's in each layer, and are the key columns complete?
2. **Permits (layer 23):** which columns mean renewal, issued, excavation and paving, and how many permits pass?
3. **Service requests (Cityworks):** does the provided SQL match the brief, and how many requests pass?
4. **Road segments (layer 162):** which road types count as streets, service lanes and ramps?
5. **Snapping:** how far are the permit points from the road?

Each section ends with the decision it led to.

*This notebook uses the Python test's environment (`python_test/.venv`, kernel "CS12 Python Test (3.11)"), since it needs the same two packages, `requests` and `pandas`. No `arcpy`, so it runs outside ArcGIS Pro.*

In [1]:
import pandas as pd
import requests

pd.set_option("display.width", 160)

PERMITS_23 = "https://maps2.dcgis.dc.gov/dcgis/rest/services/DCGIS_APPS/DDOT_TOPS/MapServer/23"
REQUESTS_CW = "https://maps2.dcgis.dc.gov/dcgis/rest/services/DDOT/Cityworks/MapServer/1"
SUBBLOCKS_162 = "https://maps2.dcgis.dc.gov/dcgis/rest/services/DCGIS_DATA/Transportation_WebMercator/MapServer/162"

## 1. Load the data
 The services hand out at most 2,000 rows per request, so it asks page by page.

- **Permits:** the whole layer (about 55,000 rows), so nothing is hidden before I decide how to filter.
- **Service requests:** Cityworks holds about 1.5 million requests going back years. The brief only wants requests created on or after 1 October 2025, so only those are downloaded (about 75,000). Every other filter is decided from the data.
- **Road segments:** the whole layer (about 35,000), without the line shapes for now.

In [2]:
def download(layer_url, where="1=1"):
    # The layer's own description: column names, types and code lists
    layer_info = requests.get(layer_url, params={"f": "json"}, timeout=60).json()

    # Ask for the rows page by page until the service says there are no more
    rows, offset = [], 0
    while True:
        params = {
            "where": where,
            "outFields": "*",
            "returnGeometry": "false",    # the points have their coordinates as columns
            "orderByFields": "OBJECTID",  # same order every time, so pages don't overlap
            "resultOffset": offset,
            "resultRecordCount": 2000,
            "f": "json",
        }
        page = requests.get(f"{layer_url}/query", params=params, timeout=120).json()
        if "error" in page:               # a bad query still comes back as "OK"
            raise RuntimeError(page["error"])
        rows += [feature["attributes"] for feature in page["features"]]
        if not page.get("exceededTransferLimit"):
            break
        offset += len(page["features"])
    table = pd.DataFrame(rows)

    # Dates arrive as milliseconds since 1970 in UTC, so convert them to DC time.
    # Coded columns get an extra "_label" column with the readable value.
    for field in layer_info["fields"]:
        name = field["name"]
        if name not in table:
            continue
        if field["type"] == "esriFieldTypeDate":
            table[name] = pd.to_datetime(table[name], unit="ms", utc=True).dt.tz_convert("America/New_York")
        domain = field.get("domain") or {}
        if domain.get("type") == "codedValue":
            labels = {str(c["code"]): c["name"] for c in domain["codedValues"]}
            table[name + "_label"] = table[name].astype("string").str.removesuffix(".0").map(labels)
    return table

In [3]:
permits = download(PERMITS_23)
service_requests = download(REQUESTS_CW, where="INITIATEDDATE >= timestamp '2025-10-01 00:00:00'")
subblocks = download(SUBBLOCKS_162)

for name, table in [("Permits (layer 23)", permits), ("Service requests since 1 Oct 2025", service_requests),
                    ("Road segments (layer 162)", subblocks)]:
    print(f"{name}: {len(table):,} rows, {table.shape[1]} columns")

Permits (layer 23): 54,625 rows, 102 columns
Service requests since 1 Oct 2025: 74,944 rows, 57 columns
Road segments (layer 162): 34,964 rows, 113 columns


In [4]:
# How complete are the columns this task depends on? (share of rows filled)
permit_columns = ["TrackingNumber", "PermitNumber", "StatusDescription", "IssueDate", "ExpirationDate",
                  "Renewal", "RenewalDescription", "IsExcavation", "IsPaving", "PermitTypeDescription",
                  "WorkLocationFullAddress", "XCoord", "YCoord", "OnSegX", "OnSegY", "StreetSegID", "RoadwaySegID"]
request_columns = ["REQUESTID", "DESCRIPTION", "STATUS", "INITIATEDDATE", "INSPECTIONDATE", "INSPECTIONCOMPLETE",
                   "CLOSEDDATE", "ADDRESS", "XCOORD", "YCOORD", "ONSEGX", "ONSEGY"]
subblock_columns = ["SUBBLOCKKEY", "ROUTENAME", "ROADTYPE"]

pd.concat([
    permits[permit_columns].notna().mean().round(3).rename("share filled").to_frame().assign(layer="permits"),
    service_requests[request_columns].notna().mean().round(3).rename("share filled").to_frame().assign(layer="service requests"),
    subblocks[subblock_columns].notna().mean().round(3).rename("share filled").to_frame().assign(layer="road segments"),
])

,share filled,layer
TrackingNumber,1.000,permits
PermitNumber,0.659,permits
StatusDescription,1.000,permits
IssueDate,0.659,permits
ExpirationDate,1.000,permits
Renewal,1.000,permits
RenewalDescription,1.000,permits
IsExcavation,1.000,permits
IsPaving,1.000,permits
PermitTypeDescription,0.866,permits


**What this shows**

- All three layers download cleanly: 54,625 permits, 74,944 service requests since 1 October 2025, and 34,964 road segments.
- The columns the filters need are always filled: `ExpirationDate`, `Renewal`, `IsExcavation` and `IsPaving` for permits; `DESCRIPTION`, `STATUS`, `INITIATEDDATE` and `INSPECTIONCOMPLETE` for requests; `ROADTYPE` for road segments.
- `IssueDate` and `PermitNumber` are filled on 66% of permits. As in layer 24, they probably only exist once a permit is issued, which section 2 checks.
- Permits can't be joined to road segments by an ID: `RoadwaySegID` is always empty and `StreetSegID` is filled on only 59%, and neither is the road segment key (`SUBBLOCKKEY`).

**Decision:** permits and requests are linked to road segments **by location** (snapping and a spatial join in ArcGIS Pro), not by a shared ID.

## 2. Permits: which rows count

The brief's three permit filters: expiration on or after 1 October 2025, only *renewal* permits that have been *issued*, and only excavation and paving permits. For each one, I check how it shows up in the data.

In [5]:
# Renewal: what do the codes mean?
pd.crosstab(permits["Renewal"], permits["RenewalDescription"])

RenewalDescription,New,Renewal
Renewal,,
1,27259,0
2,0,27366


`Renewal` is a code, and **2 means Renewal** (1 means New). So the filter is `Renewal = 2`, not `Renewal = 1`.

In [6]:
# Issued: which statuses have an issue date?
pd.crosstab(permits["StatusDescription"], permits["IssueDate"].notna().rename("has issue date"), margins=True)

has issue date,False,True,All
StatusDescription,,,
Approved (Pending Payment),11804,0,11804
Assigned,1386,0,1386
Cancel/Withdrawn,1138,0,1138
Denied,73,0,73
Issued,0,21452,21452
Not Paid,970,0,970
Pending Assignment,44,0,44
Permit Expired,23,14560,14583
Reopened,4,0,4


An issued permit keeps its issue date when its status moves on. Every "Issued" permit has an issue date, but so do 14,560 "Permit Expired" permits (issued, then expired) and a few "Suspended" and "Revoked" ones. The 23 "Permit Expired" rows without an issue date expired before they were ever issued.

**Decision:** "has been issued" = **has an issue date** (`IssueDate IS NOT NULL`), whatever the current status. Filtering on `StatusDescription = 'Issued'` alone would drop every issued permit that has since expired.

In [7]:
# Excavation and paving: how do the two flags combine?
print(pd.crosstab(permits["IsExcavation"], permits["IsPaving"]))

# What are the permits that are neither?
neither = permits[(permits["IsExcavation"] == 0) & (permits["IsPaving"] == 0)]
neither["TypeDetailNames"].str.split(";").str[0].value_counts().head(8)

IsPaving          0    1
IsExcavation            
0              1328  943
1             52222  132


TypeDetailNames
Over Head Work    676
Fixture           438
Projections       112
Landscaping        70
Mega Projects      30
Annual Permits      1
PS Rental           1
Name: count, dtype: int64

Most permits are excavation (96%), a few are paving, and only a handful are both. The permits that are neither are other work types such as overhead work and fixtures.

**Decision:** "excavation and paving permits" = **either flag** (`IsExcavation = 1 OR IsPaving = 1`). Requiring both would leave almost nothing.

In [8]:
# Expiration: what range of dates does the layer hold?
print("Earliest expiration:", permits["ExpirationDate"].min())
print("Latest expiration:  ", permits["ExpirationDate"].max())

start = pd.Timestamp("2025-10-01", tz="America/New_York")
print(f"Expiring on/after 1 Oct 2025: {(permits['ExpirationDate'] >= start).mean():.0%}")

Earliest expiration: 2026-08-31 04:00:00-04:00
Latest expiration:   2039-05-05 05:00:00-04:00
Expiring on/after 1 Oct 2025: 100%


Every permit in the layer expires after 1 October 2025. As with the occupancy permits in the Python test, the layer seems to drop permits once they have expired for a while. So the date filter is kept as the brief asks, but removes nothing in this pull.

In [9]:
# All filters together: how many rows and permits are left after each one?
steps = {
    "All rows": pd.Series(True, index=permits.index),
    "Expires on/after 1 Oct 2025": permits["ExpirationDate"] >= start,
    "Renewal (Renewal = 2)": permits["Renewal"] == 2,
    "Issued (has an issue date)": permits["IssueDate"].notna(),
    "Excavation or paving": (permits["IsExcavation"] == 1) | (permits["IsPaving"] == 1),
}
keep = pd.Series(True, index=permits.index)
counts = []
for step, rule in steps.items():
    keep &= rule
    counts.append({"filter": step, "rows": keep.sum(), "permits": permits.loc[keep, "TrackingNumber"].nunique()})
print(pd.DataFrame(counts).to_string(index=False))

filtered_permits = permits[keep]
print()
print(filtered_permits["StatusDescription"].value_counts())
print()
print("Rows per permit:", filtered_permits.groupby("TrackingNumber").size().describe().round(1).to_dict())

                     filter  rows  permits
                   All rows 54625    10191
Expires on/after 1 Oct 2025 54625    10191
      Renewal (Renewal = 2) 27366     4547
 Issued (has an issue date) 19167     3814
       Excavation or paving 18772     3552

StatusDescription
Permit Expired    10837
Issued             7934
Suspended             1
Name: count, dtype: int64

Rows per permit: {'count': 3552.0, 'mean': 5.3, 'std': 10.3, 'min': 1.0, '25%': 1.0, '50%': 1.0, '75%': 4.0, 'max': 176.0}


**What this shows**

- After all filters: about 18,800 rows from about 3,550 permits. Note that 58% of them now have the status "Permit Expired", which confirms the issue-date rule matters.
- **One permit can have many rows**, one per location (median 1, up to 176). This matters for "count of permits on each subblock": a permit with 10 locations on the same block should probably count once, not 10 times. Section 5 looks at this once the points are matched to blocks.

**Decision: the permit definition query for ArcGIS Pro**

```sql
ExpirationDate >= timestamp '2025-10-01 00:00:00'
AND Renewal = 2
AND IssueDate IS NOT NULL
AND (IsExcavation = 1 OR IsPaving = 1)
```